In [1]:

#My engine”.

#config.py
#Stores constants

DOCS_PATH = "data/fintech_docs"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL = "google/flan-t5-small"

#load_data.py

import os

def load_documents(path):
   docs = []
   for file in os.listdir(path):
       with open(os.path.join(path, file), "r", encoding="utf-8") as f:
           text = f.read()
           paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
           for i, p in enumerate(paragraphs):
               docs.append({"file": file, "chunk": i, "text": p})
   return docs

#embeddings.py

from sentence_transformers import SentenceTransformer

def load_embedder(model_name):
   return SentenceTransformer(model_name)

def encode_text(embedder, texts):
   return embedder.encode(texts, convert_to_tensor=True)

# retrieval.py
from sentence_transformers import util
import torch

def retrieve(query, embedder, docs, doc_embeddings, top_k=3):
   q_emb = embedder.encode(query, convert_to_tensor=True)
   scores = util.cos_sim(q_emb, doc_embeddings)[0]

   top_idx = torch.topk(scores, k=top_k).indices.tolist()

   results = []
   for i in top_idx:
       results.append({**docs[i], "score": float(scores[i])})

   return results

#generation.py

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

def load_generator(model_name):
   tokenizer = AutoTokenizer.from_pretrained(model_name)
   model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
   return tokenizer, model

def generate_answer(question, context, tokenizer, model):
   prompt = f"""
   Answer using only this context:
   {context}

   Question: {question}
   """

   inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
   outputs = model.generate(**inputs, max_new_tokens=200)

   return tokenizer.decode(outputs[0], skip_special_tokens=True)

#rag_pipeline.py

from src.load_data import load_documents
from src.embeddings import load_embedder, encode_text
from src.retrieval import retrieve
from src.generation import generate_answer

def run_rag(query, docs_path, embed_model, gen_model):
   docs = load_documents(docs_path)

   embedder = load_embedder(embed_model)
   texts = [d["text"] for d in docs]
   embeddings = encode_text(embedder, texts)

   retrieved = retrieve(query, embedder, docs, embeddings)

   context = "\n\n".join([r["text"] for r in retrieved])

   return retrieved, context

# app/gradio_app.py

import gradio as gr
from src.rag_pipeline import run_rag

def answer_question(q):
   retrieved, context = run_rag(
       q,
       "data/fintech_docs",
       "sentence-transformers/all-MiniLM-L6-v2",
       "google/flan-t5-small"
   )
   return context

gr.Interface(fn=answer_question, inputs="text", outputs="text").launch()




KeyboardInterrupt: 